# Pandas A to Z — The Complete Reference Notebook

A single-notebook reference covering Pandas from first principles to the
techniques used day-to-day in AI/ML/DL data pipelines: data structures,
I/O, indexing, cleaning, reshaping, grouping, time series, string
operations, performance, and plotting.

**How to use this notebook**
- Read the markdown cell before each code block — it explains *why*, not
  just *what*.
- Run cells top to bottom the first time; sample data is created inline
  so the notebook is fully self-contained (no external files required).
- Treat this as a reference you can search (Ctrl+F) rather than something
  you read once and discard.

**Table of Contents**

1. Setup and Core Data Structures
2. Creating Series and DataFrames
3. Reading and Writing Data (I/O)
4. Inspecting and Exploring Data
5. Selection and Indexing (`loc`, `iloc`, boolean masks, `query`)
6. Editing Data (adding, dropping, renaming, casting types)
7. Handling Missing Data
8. Duplicates and Data Cleaning
9. Sorting and Ranking
10. Vectorized Operations, `map`, `apply`, `applymap`
11. GroupBy — Split, Apply, Combine
12. Aggregation, Pivot Tables, and Cross-Tabulation
13. Merging, Joining, and Concatenating
14. Reshaping: `melt`, `pivot`, `stack`/`unstack`
15. MultiIndex (Hierarchical Indexing)
16. String Operations (`.str` accessor)
17. Date and Time Handling (`.dt` accessor, resampling)
18. Categorical Data
19. Window Functions (`rolling`, `expanding`, `ewm`)
20. Combining and Updating DataFrames
21. Performance Optimization
22. Plotting with Pandas
23. Styling DataFrames
24. Options, Settings, and Display Configuration
25. Quick Reference Cheat-Sheet


## 1. Setup and Core Data Structures

Pandas is built on two core data structures:

- **`Series`** — a one-dimensional labeled array. Think of it as a single
  column of a spreadsheet, with an index attached to every value.
- **`DataFrame`** — a two-dimensional labeled table made of rows and
  columns. Internally, a DataFrame is a collection of Series that share
  the same index.

Everything else in Pandas — grouping, merging, reshaping — is built on
top of these two objects.


In [1]:
import pandas as pd
import numpy as np

print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)

Pandas version: 2.2.3
NumPy version: 2.1.3


In [2]:
# A Series: one-dimensional, labeled
s = pd.Series([10, 20, 30, 40], index=['a', 'b', 'c', 'd'], name='sample_series')
s


,sample_series
a,10
b,20
c,30
d,40


In [3]:
# Every Series has an index (labels) and values (the underlying NumPy array)
print("Values:", s.values)  # This will give all values
print("Index:", s.index.tolist())
print("Dtype:", s.dtype)
print("Name:", s.name)

Values: [10 20 30 40]
Index: ['a', 'b', 'c', 'd']
Dtype: int64
Name: sample_series


In [4]:
# A DataFrame: two-dimensional, labeled rows and columns
df = pd.DataFrame({
    'gesture': ['Hello', 'Thanks', 'Yes', 'No'],
    'confidence': [0.98, 0.91, 0.87, 0.95],
    'frames_used': [15, 18, 12, 14]
})
df

,gesture,confidence,frames_used
0,Hello,0.98,15
1,Thanks,0.91,18
2,Yes,0.87,12
3,No,0.95,14


In [5]:
# Structural attributes every DataFrame exposes
print("Shape (rows, cols):", df.shape)
print("Columns:", df.columns.tolist())
print("Index:", df.index.tolist())
print("Dtypes:\n", df.dtypes)
print("Number of dimensions:", df.ndim)
print("Total elements:", df.size)

Shape (rows, cols): (4, 3)
Columns: ['gesture', 'confidence', 'frames_used']
Index: [0, 1, 2, 3]
Dtypes:
 gesture         object
confidence     float64
frames_used      int64
dtype: object
Number of dimensions: 2
Total elements: 12


## 2. Creating Series and DataFrames

DataFrames can be constructed from almost any Python data structure. Knowing
all the entry points matters because real data rarely arrives as a clean
dict — it comes from APIs (lists of dicts), NumPy arrays (model outputs),
or existing DataFrames that need reshaping.


In [9]:
# From a dictionary of lists (most common)
df1 = pd.DataFrame({'a': [1, 2, 3], 'b': [4, 5, 6]})
print(df1)
# From a list of dictionaries (common when reading JSON/API responses)
df2 = pd.DataFrame([{'a': 1, 'b': 4}, {'a': 2, 'b': 5}, {'a': 3, 'b': 6}])
print(df2)
# From a NumPy array, with explicit column names
df3 = pd.DataFrame(np.arange(6).reshape(3, 2), columns=['a', 'b'])
print(df3)
# From a list of tuples
df4 = pd.DataFrame([(1, 4), (2, 5), (3, 6)], columns=['a', 'b'])
print(df4)
# All four are equivalent
print(df1.equals(df2), df1.equals(df3), df1.equals(df4))

   a  b
0  1  4
1  2  5
2  3  6
   a  b
0  1  4
1  2  5
2  3  6
   a  b
0  0  1
1  2  3
2  4  5
   a  b
0  1  4
1  2  5
2  3  6
True False True


In [10]:
# Empty DataFrame with a defined schema — useful when you build rows in a loop
schema_df = pd.DataFrame(columns=['gesture_id', 'label', 'timestamp'])
schema_df = schema_df.astype({'gesture_id': 'int64', 'label': 'string', 'timestamp': 'datetime64[ns]'})
schema_df.dtypes

,0
gesture_id,int64
label,string[python]
timestamp,datetime64[ns]


In [12]:
# Series -> DataFrame, and DataFrame -> Series (single column)
labels = pd.Series(['A', 'B', 'C'], name='label')
as_df = labels.to_frame()
back_to_series = as_df['label']
print(type(labels), type(as_df), type(back_to_series))

<class 'pandas.core.series.Series'> <class 'pandas.core.frame.DataFrame'> <class 'pandas.core.series.Series'>


In [13]:
# copy vs reference — a subtle but important trap.
# Selecting a column returns a VIEW in some pandas versions and a COPY in others.
# Never rely on it implicitly: use .copy() when you intend to mutate independently.
df_original = pd.DataFrame({'x': [1, 2, 3]})
df_safe_copy = df_original.copy()
df_safe_copy['x'] = df_safe_copy['x'] * 100
print("Original untouched:", df_original['x'].tolist())
print("Copy modified:", df_safe_copy['x'].tolist())

Original untouched: [1, 2, 3]
Copy modified: [100, 200, 300]


## 3. Reading and Writing Data (I/O)

Pandas supports a wide range of formats. `read_csv` is the one you will use
most, but for ML pipelines `read_parquet` is usually the better default
for large datasets — it preserves dtypes and is far more compact and faster
to read than CSV.


In [14]:
# Build a small sample CSV in memory so this notebook is self-contained
from io import StringIO

csv_text = """gesture,confidence,frames_used,recorded_at
Hello,0.98,15,2025-01-01 10:00:00
Thanks,0.91,18,2025-01-01 10:01:00
Yes,0.87,12,2025-01-01 10:02:00
No,0.95,14,2025-01-01 10:03:00
"""

df_csv = pd.read_csv(StringIO(csv_text), parse_dates=['recorded_at'])
df_csv

,gesture,confidence,frames_used,recorded_at
0,Hello,0.98,15,2025-01-01 10:00:00
1,Thanks,0.91,18,2025-01-01 10:01:00
2,Yes,0.87,12,2025-01-01 10:02:00
3,No,0.95,14,2025-01-01 10:03:00


In [19]:
# Common read_csv arguments worth knowing (not all used above):
# - sep / delimiter      : field separator (default ',')
# - header                : row number(s) to use as column names
# - names                 : explicit column names if file has no header
# - index_col             : column(s) to use as the row index
# - usecols               : read only a subset of columns (saves memory)
# - dtype                 : force specific dtypes on read (faster, avoids inference bugs)
# - parse_dates           : columns to parse as datetime
# - na_values             : extra strings to treat as NaN
# - nrows                 : read only the first N rows (quick previews of huge files)
# - chunksize             : return an iterator of DataFrames for out-of-core processing

df_typed = pd.read_csv(
    StringIO(csv_text),
    usecols=['gesture', 'confidence'],
    dtype={'gesture': 'string', 'confidence': 'float32'}
)
df_typed.dtypes

,0
gesture,string[python]
confidence,float32


In [22]:
# Writing back out
df_csv.to_csv('sample_output.csv', index=False)

# Reading it back to confirm round-trip integrity
pd.read_csv('sample_output.csv').head()

,gesture,confidence,frames_used,recorded_at
0,Hello,0.98,15,2025-01-01 10:00:00
1,Thanks,0.91,18,2025-01-01 10:01:00
2,Yes,0.87,12,2025-01-01 10:02:00
3,No,0.95,14,2025-01-01 10:03:00


In [23]:
# Excel (requires openpyxl) — common for handing results to non-technical stakeholders
df_csv.to_excel('sample_output.xlsx', index=False, sheet_name='gestures')
pd.read_excel('sample_output.xlsx', sheet_name='gestures').head()

,gesture,confidence,frames_used,recorded_at
0,Hello,0.98,15,2025-01-01 10:00:00
1,Thanks,0.91,18,2025-01-01 10:01:00
2,Yes,0.87,12,2025-01-01 10:02:00
3,No,0.95,14,2025-01-01 10:03:00


In [24]:
# JSON — common for API payloads and config-like data
df_csv.to_json('sample_output.json', orient='records', date_format='iso')
pd.read_json('sample_output.json')

,gesture,confidence,frames_used,recorded_at
0,Hello,0.98,15,2025-01-01 10:00:00
1,Thanks,0.91,18,2025-01-01 10:01:00
2,Yes,0.87,12,2025-01-01 10:02:00
3,No,0.95,14,2025-01-01 10:03:00


In [25]:
# Parquet — columnar, compressed, dtype-preserving. Preferred for ML datasets.
# Requires pyarrow or fastparquet installed.
try:
    df_csv.to_parquet('sample_output.parquet', index=False)
    print(pd.read_parquet('sample_output.parquet').dtypes)
except ImportError as e:
    print("Install pyarrow to use parquet:", e)

gesture                object
confidence            float64
frames_used             int64
recorded_at    datetime64[ns]
dtype: object


In [26]:
# Reading very large files without loading everything into memory: chunksize
total_rows = 0
for chunk in pd.read_csv(StringIO(csv_text), chunksize=2):
    total_rows += len(chunk)
print("Processed rows via chunking:", total_rows)

# SQL (pattern only — requires a real connection, e.g. sqlite3/SQLAlchemy engine):
# import sqlite3
# conn = sqlite3.connect('database.db')
# df = pd.read_sql('SELECT * FROM gestures', conn)
# df.to_sql('gestures', conn, if_exists='replace', index=False)

Processed rows via chunking: 4


## 4. Inspecting and Exploring Data

The first thing to run on any new dataset — before any modeling — is a
quick structural and statistical inspection. This catches wrong dtypes,
unexpected nulls, and outliers early.


In [27]:
df = pd.DataFrame({
    'gesture': ['Hello', 'Thanks', 'Yes', 'No', 'Hello', 'Please', None],
    'confidence': [0.98, 0.91, 0.87, 0.95, 0.96, np.nan, 0.80],
    'frames_used': [15, 18, 12, 14, 16, 20, 11],
    'session_id': [1, 1, 1, 2, 2, 2, 3]
})

df.head(3)      # first n rows (default 5)


,gesture,confidence,frames_used,session_id
0,Hello,0.98,15,1
1,Thanks,0.91,18,1
2,Yes,0.87,12,1


In [28]:
df.tail(2)      # last n rows

,gesture,confidence,frames_used,session_id
5,Please,NaN,20,2
6,None,0.8,11,3


In [29]:
df.info()       # dtypes, non-null counts, memory usage — always run this first

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   gesture      6 non-null      object 
 1   confidence   6 non-null      float64
 2   frames_used  7 non-null      int64  
 3   session_id   7 non-null      int64  
dtypes: float64(1), int64(2), object(1)
memory usage: 356.0+ bytes


In [30]:
df.describe()   # summary statistics for numeric columns

,confidence,frames_used,session_id
count,6.000000,7.000000,7.000000
mean,0.911667,15.142857,1.714286
std,0.067355,3.184785,0.755929
min,0.800000,11.000000,1.000000
25%,0.880000,13.000000,1.000000
50%,0.930000,15.000000,2.000000
75%,0.957500,17.000000,2.000000
max,0.980000,20.000000,3.000000


In [31]:
df.describe(include='all')   # include non-numeric columns too (counts, unique, top, freq)

,gesture,confidence,frames_used,session_id
count,6,6.000000,7.000000,7.000000
unique,5,NaN,NaN,NaN
top,Hello,NaN,NaN,NaN
freq,2,NaN,NaN,NaN
mean,NaN,0.911667,15.142857,1.714286
std,NaN,0.067355,3.184785,0.755929
min,NaN,0.800000,11.000000,1.000000
25%,NaN,0.880000,13.000000,1.000000
50%,NaN,0.930000,15.000000,2.000000
75%,NaN,0.957500,17.000000,2.000000


In [33]:
print("Shape:", df.shape)
print("Null counts per column:\n", df.isna().sum())
print("Unique gestures:", df['gesture'].nunique())
print("Value counts:\n", df['gesture'].value_counts(dropna=False))
print("Memory usage (bytes):\n", df.memory_usage(deep=True))

Shape: (7, 4)
Null counts per column:
 gesture        1
confidence     1
frames_used    0
session_id     0
dtype: int64
Unique gestures: 5
Value counts:
 gesture
Hello     2
Thanks    1
Yes       1
No        1
Please    1
None      1
Name: count, dtype: int64
Memory usage (bytes):
 Index          132
gesture        345
confidence      56
frames_used     56
session_id      56
dtype: int64


In [34]:
# sample() — random rows, useful for spot-checking large datasets instead of always
# looking at head()/tail() which can hide problems in the middle of the data
df.sample(n=3, random_state=42)

,gesture,confidence,frames_used,session_id
0,Hello,0.98,15,1
1,Thanks,0.91,18,1
5,Please,NaN,20,2


## 5. Selection and Indexing

This is the single most important section to master. Getting selection
wrong is the most common source of subtle bugs (`SettingWithCopyWarning`,
silently wrong slices).

**Rule of thumb**
- `df['col']` / `df[['col1','col2']]` — select columns
- `df.loc[row_labels, col_labels]` — select by **label**
- `df.iloc[row_positions, col_positions]` — select by **integer position**
- Boolean masks — select rows matching a condition
- `df.query(...)` — SQL-like string-based filtering


In [35]:
df = pd.DataFrame({
    'gesture': ['Hello', 'Thanks', 'Yes', 'No', 'Please'],
    'confidence': [0.98, 0.91, 0.87, 0.95, 0.99],
    'frames_used': [15, 18, 12, 14, 20]
}, index=['g1', 'g2', 'g3', 'g4', 'g5'])

df['gesture']              # single column -> Series

,gesture
g1,Hello
g2,Thanks
g3,Yes
g4,No
g5,Please


In [36]:
df[['gesture', 'confidence']]     # multiple columns -> DataFrame

,gesture,confidence
g1,Hello,0.98
g2,Thanks,0.91
g3,Yes,0.87
g4,No,0.95
g5,Please,0.99


In [37]:
# .loc — label based. Note: end label IS included (unlike Python slicing).
df.loc['g1':'g3', ['gesture', 'confidence']]

,gesture,confidence
g1,Hello,0.98
g2,Thanks,0.91
g3,Yes,0.87


In [38]:
# .iloc — integer position based. Note: end position is EXCLUDED, like normal Python slicing.
df.iloc[0:3, 0:2]

,gesture,confidence
g1,Hello,0.98
g2,Thanks,0.91
g3,Yes,0.87


In [39]:
# .at / .iat — fast scalar access (label / position respectively). Use when you need ONE value.
print(df.at['g1', 'confidence'])
print(df.iat[0, 1])

0.98
0.98


In [40]:
# Boolean indexing — the most common filtering pattern
high_confidence = df[df['confidence'] > 0.95]
high_confidence

,gesture,confidence,frames_used
g1,Hello,0.98,15
g5,Please,0.99,20


In [41]:
# Combining multiple conditions: use & / | / ~ with PARENTHESES (not 'and'/'or')
mask = (df['confidence'] > 0.9) & (df['frames_used'] < 18)
df[mask]

,gesture,confidence,frames_used
g1,Hello,0.98,15
g4,No,0.95,14


In [42]:
# .isin() for membership tests
df[df['gesture'].isin(['Hello', 'Yes'])]

,gesture,confidence,frames_used
g1,Hello,0.98,15
g3,Yes,0.87,12


In [43]:
# .query() — often more readable for complex conditions
df.query('confidence > 0.9 and frames_used < 18')

,gesture,confidence,frames_used
g1,Hello,0.98,15
g4,No,0.95,14


In [44]:
# Setting values safely — always assign through .loc, never chained indexing
df.loc[df['gesture'] == 'Hello', 'confidence'] = 0.99

# ANTI-PATTERN (avoid): df[df['gesture'] == 'Hello']['confidence'] = 0.99
# This chains two selections and pandas cannot guarantee it modifies the
# original DataFrame -> triggers SettingWithCopyWarning and may silently fail.
df

,gesture,confidence,frames_used
g1,Hello,0.99,15
g2,Thanks,0.91,18
g3,Yes,0.87,12
g4,No,0.95,14
g5,Please,0.99,20


## 6. Editing Data — Adding, Dropping, Renaming, Casting Types

Correct dtypes matter enormously for ML pipelines: a numeric column stored
as `object` (string) will silently break scikit-learn/TensorFlow input
pipelines, and won't be caught until training fails.


In [45]:
df = pd.DataFrame({
    'gesture': ['Hello', 'Thanks', 'Yes'],
    'confidence': [0.98, 0.91, 0.87]
})

# Adding a new column
df['is_confident'] = df['confidence'] > 0.9

# Adding a computed column from existing ones
df['confidence_pct'] = (df['confidence'] * 100).round(1)
df

,gesture,confidence,is_confident,confidence_pct
0,Hello,0.98,True,98.0
1,Thanks,0.91,True,91.0
2,Yes,0.87,False,87.0


In [46]:
# insert() — control WHERE the new column goes (default assignment appends at the end)
df.insert(loc=1, column='gesture_length', value=df['gesture'].str.len())
df

,gesture,gesture_length,confidence,is_confident,confidence_pct
0,Hello,5,0.98,True,98.0
1,Thanks,6,0.91,True,91.0
2,Yes,3,0.87,False,87.0


In [47]:
# Dropping columns / rows
df_dropped_col = df.drop(columns=['gesture_length'])
df_dropped_row = df.drop(index=0)
print(df_dropped_col.columns.tolist())
print(df_dropped_row.index.tolist())

# drop() returns a NEW object by default; pass inplace=True to mutate in place
# (inplace is generally discouraged: it prevents method chaining and can be
# slower since pandas still often builds a new object internally)

['gesture', 'confidence', 'is_confident', 'confidence_pct']
[1, 2]


In [49]:
# Renaming columns
df_renamed = df.rename(columns={'confidence_pct': 'confidence_percent'})

# Renaming ALL columns at once
df_relabeled = df.copy()
df_relabeled.columns = ['label', 'label_len', 'conf', 'is_high_conf', 'conf_pct']
df_relabeled.head(2)

,label,label_len,conf,is_high_conf,conf_pct
0,Hello,5,0.98,True,98.0
1,Thanks,6,0.91,True,91.0


In [50]:
# Casting dtypes — the most common fix needed before feeding data into a model
df['confidence'] = df['confidence'].astype('float32')     # smaller footprint for ML
df['is_confident'] = df['is_confident'].astype('int8')    # bool -> int for some models
df.dtypes

,0
gesture,object
gesture_length,int64
confidence,float32
is_confident,int8
confidence_pct,float64


In [51]:
# to_numeric / to_datetime with error handling — critical for messy real-world data
messy = pd.Series(['0.9', '0.8', 'not_a_number', '0.95'])
cleaned = pd.to_numeric(messy, errors='coerce')   # invalid parses become NaN instead of raising
cleaned

,0
0,0.90
1,0.80
2,NaN
3,0.95


## 7. Handling Missing Data

Missing data is unavoidable in any real dataset (sensor dropout, failed
landmark detection frames, incomplete labels). How you handle it directly
affects model performance — dropping too aggressively wastes data;
filling incorrectly introduces bias.


In [52]:
df = pd.DataFrame({
    'gesture': ['Hello', 'Thanks', None, 'No', 'Please'],
    'confidence': [0.98, np.nan, 0.87, 0.95, np.nan],
    'frames_used': [15, 18, 12, np.nan, 20]
})

df.isna()          # boolean mask of missing values (isnull is an alias)

,gesture,confidence,frames_used
0,False,False,False
1,False,True,False
2,True,False,False
3,False,False,True
4,False,True,False


In [53]:
print("Missing per column:\n", df.isna().sum())
print("Rows with ANY missing value:", df.isna().any(axis=1).sum())
print("Percentage missing per column:\n", (df.isna().mean() * 100).round(1))

Missing per column:
 gesture        1
confidence     2
frames_used    1
dtype: int64
Rows with ANY missing value: 4
Percentage missing per column:
 gesture        20.0
confidence     40.0
frames_used    20.0
dtype: float64


In [54]:
# dropna — remove rows/columns with missing data
df.dropna()                       # drop rows with ANY NaN

,gesture,confidence,frames_used
0,Hello,0.98,15.0


In [55]:
df.dropna(subset=['gesture'])     # drop rows only if THIS column is NaN

,gesture,confidence,frames_used
0,Hello,0.98,15.0
1,Thanks,NaN,18.0
3,No,0.95,NaN
4,Please,NaN,20.0


In [56]:
df.dropna(thresh=2)               # keep rows with at least 2 non-null values

,gesture,confidence,frames_used
0,Hello,0.98,15.0
1,Thanks,NaN,18.0
2,None,0.87,12.0
3,No,0.95,NaN
4,Please,NaN,20.0


In [57]:
# fillna — impute missing values. The imputation strategy should match the reasoning
# below; never fill blindly.
df_filled = df.copy()
df_filled['confidence'] = df_filled['confidence'].fillna(df_filled['confidence'].mean())
df_filled['frames_used'] = df_filled['frames_used'].fillna(df_filled['frames_used'].median())
df_filled['gesture'] = df_filled['gesture'].fillna('Unknown')
df_filled

,gesture,confidence,frames_used
0,Hello,0.980000,15.0
1,Thanks,0.933333,18.0
2,Unknown,0.870000,12.0
3,No,0.950000,16.5
4,Please,0.933333,20.0


In [58]:
# Forward-fill / backward-fill — appropriate for ordered/time-series data,
# NOT for i.i.d. tabular rows (it assumes temporal/positional continuity)
ts = pd.Series([1.0, np.nan, np.nan, 4.0, np.nan])
print("ffill:", ts.ffill().tolist())
print("bfill:", ts.bfill().tolist())
print("interpolate (linear):", ts.interpolate().tolist())

ffill: [1.0, 1.0, 1.0, 4.0, 4.0]
bfill: [1.0, 4.0, 4.0, 4.0, nan]
interpolate (linear): [1.0, 2.0, 3.0, 4.0, 4.0]


**Choosing an imputation strategy — the reasoning that matters:**

| Situation | Recommended approach | Why |
|---|---|---|
| Missing completely at random, numeric | Mean/median fill | Simple, low bias if truly random |
| Missing not at random (e.g., low-light frames fail landmark detection) | Flag with an indicator column + impute | The *missingness itself* is informative — dropping it loses signal |
| Time-ordered data (sensor stream) | Forward/backward fill or interpolation | Preserves temporal continuity |
| Categorical/label column | New 'Unknown' category, or drop the row if the label is the target | Filling a *target* label is usually wrong — you'd be inventing ground truth |
| Small % missing (<5%) and no clear pattern | Drop rows | Simplicity outweighs the small data loss |
| Large % missing (>40%) in a column | Consider dropping the column entirely | Imputing a mostly-empty column adds noise, not signal |

**Risk to flag explicitly:** imputing a column that will become a training
*label* (as opposed to a feature) is generally unsound — synthetic labels
teach the model incorrect ground truth. Prefer dropping unlabeled rows.


## 8. Duplicates and Data Cleaning


In [59]:
df = pd.DataFrame({
    'gesture': ['Hello', 'Hello', 'Thanks', 'Yes', 'Yes'],
    'confidence': [0.98, 0.98, 0.91, 0.87, 0.99]
})

print(df.duplicated())              # boolean mask, True = duplicate of an earlier row
print(df.duplicated().sum(), "duplicate rows")
df.drop_duplicates()

0    False
1     True
2    False
3    False
4    False
dtype: bool
1 duplicate rows


,gesture,confidence
0,Hello,0.98
2,Thanks,0.91
3,Yes,0.87
4,Yes,0.99


In [60]:
# Duplicated based on a subset of columns, keeping the LAST occurrence instead of first
df.drop_duplicates(subset=['gesture'], keep='last')

,gesture,confidence
1,Hello,0.98
2,Thanks,0.91
4,Yes,0.99


In [61]:
# replace() — targeted value substitution
df['gesture'] = df['gesture'].replace({'Hello': 'Hi', 'Thanks': 'Thank you'})
df

,gesture,confidence
0,Hi,0.98
1,Hi,0.98
2,Thank you,0.91
3,Yes,0.87
4,Yes,0.99


In [62]:
# String cleaning is extremely common before this data reaches a model
messy = pd.Series([' Hello ', 'THANKS', 'yes!!', None])
cleaned = (
    messy
    .str.strip()
    .str.lower()
    .str.replace(r'[^a-z\s]', '', regex=True)
)
cleaned

,0
0,hello
1,thanks
2,yes
3,None


In [63]:
# clip() — cap outlier values into a valid range (e.g. confidence scores must be in [0, 1])
scores = pd.Series([0.5, 1.2, -0.1, 0.95])
scores.clip(lower=0, upper=1)

,0
0,0.50
1,1.00
2,0.00
3,0.95


## 9. Sorting and Ranking


In [67]:
df = pd.DataFrame({
    'gesture': ['Hello', 'Thanks', 'Yes', 'No'],
    'confidence': [0.98, 0.91, 0.87, 0.95]
})

df.sort_values('confidence')                       # ascending by default

,gesture,confidence
2,Yes,0.87
1,Thanks,0.91
3,No,0.95
0,Hello,0.98


In [68]:
df.sort_values('confidence', ascending=False)       # descending

,gesture,confidence
0,Hello,0.98
3,No,0.95
1,Thanks,0.91
2,Yes,0.87


In [69]:
df.sort_values(['confidence', 'gesture'], ascending=[False, True])   # multi-column sort

,gesture,confidence
0,Hello,0.98
3,No,0.95
1,Thanks,0.91
2,Yes,0.87


In [70]:
df.sort_index()               # sort by the row index labels

# nlargest / nsmallest — faster and clearer than sort_values().head() for top-N lookups
print(df.nlargest(2, 'confidence'))
df['rank'] = df['confidence'].rank(ascending=False)
df

  gesture  confidence
0   Hello        0.98
3      No        0.95


,gesture,confidence,rank
0,Hello,0.98,1.0
1,Thanks,0.91,3.0
2,Yes,0.87,4.0
3,No,0.95,2.0


## 10. Vectorized Operations, `map`, `apply`, `applymap`

**Performance ordering, from fastest to slowest — this matters a lot on
large datasets:**

1. Built-in vectorized operations (`df['a'] + df['b']`, `.str.lower()`, `np.where`)
2. `Series.map()` for element-wise lookups/substitutions
3. `DataFrame.apply()` for row/column-wise custom logic
4. `.applymap()` (element-wise on the whole DataFrame — deprecated in favor of `.map()` on DataFrame since pandas 2.1)
5. A raw Python `for` loop — almost always avoid this

Vectorized operations use compiled C/NumPy code under the hood and can be
10–100x faster than the equivalent `.apply()` with a Python function.


In [71]:
df = pd.DataFrame({'a': [1, 2, 3, 4], 'b': [10, 20, 30, 40]})

# 1. Vectorized arithmetic — fastest, always prefer this shape of solution
df['sum'] = df['a'] + df['b']
df['scaled'] = df['a'] * 100
df

,a,b,sum,scaled
0,1,10,11,100
1,2,20,22,200
2,3,30,33,300
3,4,40,44,400


In [72]:
# np.where — vectorized conditional (like an inline if/else across a whole column)
df['category'] = np.where(df['a'] > 2, 'high', 'low')
df

,a,b,sum,scaled,category
0,1,10,11,100,low
1,2,20,22,200,low
2,3,30,33,300,high
3,4,40,44,400,high


In [73]:
# Series.map() — element-wise value substitution via dict or function
mapping = {1: 'one', 2: 'two', 3: 'three', 4: 'four'}
df['a_word'] = df['a'].map(mapping)
df

,a,b,sum,scaled,category,a_word
0,1,10,11,100,low,one
1,2,20,22,200,low,two
2,3,30,33,300,high,three
3,4,40,44,400,high,four


In [74]:
# DataFrame.apply() — custom row-wise or column-wise logic that can't be vectorized directly
def confidence_band(row):
    if row['a'] > 3:
        return 'A'
    elif row['a'] > 1:
        return 'B'
    return 'C'

df['band'] = df.apply(confidence_band, axis=1)   # axis=1 -> apply across each ROW
df

,a,b,sum,scaled,category,a_word,band
0,1,10,11,100,low,one,C
1,2,20,22,200,low,two,B
2,3,30,33,300,high,three,B
3,4,40,44,400,high,four,A


In [75]:
# apply() with axis=0 (default) applies down each COLUMN
numeric_only = df[['a', 'b']]
numeric_only.apply(lambda col: col.max() - col.min(), axis=0)

,0
a,3
b,30


In [78]:
# .map() on a DataFrame (pandas >= 2.1) replaces the deprecated .applymap()
# element-wise transform across every cell
numeric_only.map(lambda x: x ** 2)

,a,b
0,1,100
1,4,400
2,9,900
3,16,1600


## 11. GroupBy — Split, Apply, Combine

`groupby` is the pandas equivalent of SQL's `GROUP BY`. It follows a
three-step model:

1. **Split** the data into groups based on a key.
2. **Apply** a function to each group independently.
3. **Combine** the results back into a single object.


In [80]:
df = pd.DataFrame({
    'session_id': [1, 1, 1, 2, 2, 3, 3, 3],
    'gesture': ['Hello', 'Thanks', 'Hello', 'Yes', 'No', 'Hello', 'Yes', 'Yes'],
    'confidence': [0.98, 0.91, 0.95, 0.87, 0.93, 0.99, 0.85, 0.88]
})

grouped = df.groupby('session_id')
grouped['confidence'].mean()      # average confidence per session

,confidence
session_id,
1,0.946667
2,0.900000
3,0.906667


In [81]:
# Multiple aggregations at once with .agg()
grouped['confidence'].agg(['mean', 'min', 'max', 'count'])

,mean,min,max,count
session_id,,,,
1,0.946667,0.91,0.98,3
2,0.900000,0.87,0.93,2
3,0.906667,0.85,0.99,3


In [82]:
# Different aggregations for different columns
df.groupby('session_id').agg(
    avg_confidence=('confidence', 'mean'),
    gesture_count=('gesture', 'count'),
    unique_gestures=('gesture', 'nunique')
)

,avg_confidence,gesture_count,unique_gestures
session_id,,,
1,0.946667,3,2
2,0.900000,2,2
3,0.906667,3,2


In [83]:
# Grouping by multiple keys
df.groupby(['session_id', 'gesture'])['confidence'].mean()

session_id  gesture
1           Hello      0.965
            Thanks     0.910
2           No         0.930
            Yes        0.870
3           Hello      0.990
            Yes        0.865
Name: confidence, dtype: float64

In [84]:
# transform() — returns a result the SAME SHAPE as the original DataFrame,
# useful for creating new feature columns (e.g. "confidence relative to session average")
df['session_avg_confidence'] = df.groupby('session_id')['confidence'].transform('mean')
df['confidence_vs_session_avg'] = df['confidence'] - df['session_avg_confidence']
df

,session_id,gesture,confidence,session_avg_confidence,confidence_vs_session_avg
0,1,Hello,0.98,0.946667,0.033333
1,1,Thanks,0.91,0.946667,-0.036667
2,1,Hello,0.95,0.946667,0.003333
3,2,Yes,0.87,0.900000,-0.030000
4,2,No,0.93,0.900000,0.030000
5,3,Hello,0.99,0.906667,0.083333
6,3,Yes,0.85,0.906667,-0.056667
7,3,Yes,0.88,0.906667,-0.026667


In [85]:
# filter() — keep only groups that satisfy a condition (e.g. sessions with >2 gestures)
df.groupby('session_id').filter(lambda g: len(g) > 2)

,session_id,gesture,confidence,session_avg_confidence,confidence_vs_session_avg
0,1,Hello,0.98,0.946667,0.033333
1,1,Thanks,0.91,0.946667,-0.036667
2,1,Hello,0.95,0.946667,0.003333
5,3,Hello,0.99,0.906667,0.083333
6,3,Yes,0.85,0.906667,-0.056667
7,3,Yes,0.88,0.906667,-0.026667


In [86]:
# Iterating over groups directly (rarely needed, but useful for debugging)
for session_id, group in df.groupby('session_id'):
    print(f"Session {session_id}: {len(group)} gestures, avg confidence {group['confidence'].mean():.2f}")

Session 1: 3 gestures, avg confidence 0.95
Session 2: 2 gestures, avg confidence 0.90
Session 3: 3 gestures, avg confidence 0.91


## 12. Aggregation, Pivot Tables, and Cross-Tabulation


In [87]:
df = pd.DataFrame({
    'user': ['u1', 'u1', 'u2', 'u2', 'u3', 'u3'],
    'gesture': ['Hello', 'Yes', 'Hello', 'No', 'Yes', 'Yes'],
    'confidence': [0.98, 0.87, 0.95, 0.93, 0.85, 0.88]
})

# pivot_table — like an Excel pivot table: reshape + aggregate in one step
pd.pivot_table(df, values='confidence', index='user', columns='gesture', aggfunc='mean')

gesture,Hello,No,Yes
user,,,
u1,0.98,NaN,0.870
u2,0.95,0.93,NaN
u3,NaN,NaN,0.865


In [88]:
# fill_value handles the NaN cells created where a user/gesture combo never occurred
pd.pivot_table(df, values='confidence', index='user', columns='gesture',
                aggfunc='mean', fill_value=0)

gesture,Hello,No,Yes
user,,,
u1,0.98,0.00,0.870
u2,0.95,0.93,0.000
u3,0.00,0.00,0.865


In [89]:
# crosstab — frequency counts between two categorical columns (no aggregation column needed)
pd.crosstab(df['user'], df['gesture'])

gesture,Hello,No,Yes
user,,,
u1,1,0,1
u2,1,1,0
u3,0,0,2


In [90]:
# crosstab with normalization — proportions instead of raw counts
pd.crosstab(df['user'], df['gesture'], normalize='index')   # row-wise proportions

gesture,Hello,No,Yes
user,,,
u1,0.5,0.0,0.5
u2,0.5,0.5,0.0
u3,0.0,0.0,1.0


## 13. Merging, Joining, and Concatenating

- **`concat`** — stack DataFrames on top of each other (rows) or side by
  side (columns). No key matching involved.
- **`merge`** — SQL-style joins on one or more key columns.
- **`join`** — a convenience method for merging on the *index* specifically.


In [91]:
df_a = pd.DataFrame({'gesture_id': [1, 2, 3], 'label': ['Hello', 'Thanks', 'Yes']})
df_b = pd.DataFrame({'gesture_id': [1, 2, 4], 'sample_count': [120, 95, 80]})

# INNER join (default) — keep only keys present in BOTH
pd.merge(df_a, df_b, on='gesture_id', how='inner')

,gesture_id,label,sample_count
0,1,Hello,120
1,2,Thanks,95


In [92]:
# LEFT join — keep all rows from df_a, fill unmatched df_b columns with NaN
pd.merge(df_a, df_b, on='gesture_id', how='left')

,gesture_id,label,sample_count
0,1,Hello,120.0
1,2,Thanks,95.0
2,3,Yes,NaN


In [93]:
# OUTER join — keep every key from either side
pd.merge(df_a, df_b, on='gesture_id', how='outer')

,gesture_id,label,sample_count
0,1,Hello,120.0
1,2,Thanks,95.0
2,3,Yes,NaN
3,4,NaN,80.0


In [94]:
# RIGHT join — keep all rows from df_b
pd.merge(df_a, df_b, on='gesture_id', how='right')

,gesture_id,label,sample_count
0,1,Hello,120
1,2,Thanks,95
2,4,NaN,80


In [95]:
# merge with indicator=True — shows which side(s) each row came from (useful for debugging joins)
pd.merge(df_a, df_b, on='gesture_id', how='outer', indicator=True)

,gesture_id,label,sample_count,_merge
0,1,Hello,120.0,both
1,2,Thanks,95.0,both
2,3,Yes,NaN,left_only
3,4,NaN,80.0,right_only


In [96]:
# concat — stacking rows (axis=0, default). Schemas should match; mismatches produce NaN.
batch1 = pd.DataFrame({'gesture': ['Hello', 'Thanks']})
batch2 = pd.DataFrame({'gesture': ['Yes', 'No']})
pd.concat([batch1, batch2], ignore_index=True)   # ignore_index resets 0..n instead of repeating labels

,gesture
0,Hello
1,Thanks
2,Yes
3,No


In [97]:
# concat — stacking columns (axis=1). Rows are aligned by index.
left = pd.DataFrame({'gesture': ['Hello', 'Thanks']})
right = pd.DataFrame({'confidence': [0.98, 0.91]})
pd.concat([left, right], axis=1)

,gesture,confidence
0,Hello,0.98
1,Thanks,0.91


In [98]:
# join() — merge by index, a common pattern when both frames were derived
# from the same original index (e.g. separate feature extraction steps)
features = pd.DataFrame({'landmark_var': [0.02, 0.05]}, index=['g1', 'g2'])
labels = pd.DataFrame({'label': ['Hello', 'Thanks']}, index=['g1', 'g2'])
features.join(labels)

,landmark_var,label
g1,0.02,Hello
g2,0.05,Thanks


## 14. Reshaping: `melt`, `pivot`, `stack` / `unstack`

- **Wide format**: one row per entity, one column per variable (common for reports).
- **Long/tidy format**: one row per (entity, variable, value) triple (what most
  ML and plotting libraries expect).

`melt` converts wide → long. `pivot` converts long → wide.


In [99]:
wide = pd.DataFrame({
    'user': ['u1', 'u2'],
    'Hello_conf': [0.98, 0.95],
    'Thanks_conf': [0.91, 0.89]
})
wide

,user,Hello_conf,Thanks_conf
0,u1,0.98,0.91
1,u2,0.95,0.89


In [100]:
# melt: wide -> long
long = wide.melt(id_vars='user', var_name='gesture_metric', value_name='confidence')
long

,user,gesture_metric,confidence
0,u1,Hello_conf,0.98
1,u2,Hello_conf,0.95
2,u1,Thanks_conf,0.91
3,u2,Thanks_conf,0.89


In [101]:
# pivot: long -> wide (the inverse of melt). Requires unique index/column combinations.
long['gesture'] = long['gesture_metric'].str.replace('_conf', '', regex=False)
long.pivot(index='user', columns='gesture', values='confidence')

gesture,Hello,Thanks
user,,
u1,0.98,0.91
u2,0.95,0.89


In [102]:
# stack / unstack operate on MultiIndex column <-> row levels
multi_col = pd.DataFrame(
    {('Hello', 'confidence'): [0.98, 0.95], ('Hello', 'frames'): [15, 14]},
)
multi_col.columns = pd.MultiIndex.from_tuples(multi_col.columns)
stacked = multi_col.stack(future_stack=True)   # moves innermost column level into the row index
stacked

Hello
0 confidence   0.98
  frames      15.00
1 confidence   0.95
  frames      14.00

## 15. MultiIndex (Hierarchical Indexing)

A MultiIndex lets a single axis (rows or columns) represent more than one
level of labeling — e.g. (session, gesture) as a combined row key. This is
what `groupby` with multiple keys produces automatically.


In [103]:
arrays = [
    ['s1', 's1', 's2', 's2'],
    ['Hello', 'Thanks', 'Hello', 'Yes']
]
index = pd.MultiIndex.from_arrays(arrays, names=['session', 'gesture'])
df = pd.DataFrame({'confidence': [0.98, 0.91, 0.95, 0.87]}, index=index)
df

confidence
session gesture            
s1      Hello          0.98
        Thanks         0.91
s2      Hello          0.95
        Yes            0.87

In [104]:
# Selecting from a MultiIndex
print(df.loc['s1'])                     # all rows under session s1
print(df.loc[('s1', 'Hello')])          # a specific (session, gesture) pair

         confidence
gesture            
Hello          0.98
Thanks         0.91
confidence    0.98
Name: (s1, Hello), dtype: float64


In [105]:
# xs() — cross-section, select at a specific level without needing the full key tuple
df.xs('Hello', level='gesture')

,confidence
session,
s1,0.98
s2,0.95


In [106]:
# Flattening a MultiIndex back to a normal column-based DataFrame
df.reset_index()

,session,gesture,confidence
0,s1,Hello,0.98
1,s1,Thanks,0.91
2,s2,Hello,0.95
3,s2,Yes,0.87


In [107]:
# Setting a MultiIndex from existing columns
flat = df.reset_index()
flat.set_index(['session', 'gesture'])

confidence
session gesture            
s1      Hello          0.98
        Thanks         0.91
s2      Hello          0.95
        Yes            0.87

## 16. String Operations (`.str` accessor)

The `.str` accessor applies vectorized string methods across an entire
Series without writing a manual loop — essential for cleaning text labels,
filenames, or transcript data.


In [108]:
s = pd.Series([' Hello_World ', 'THANKS-you', 'yes!!', None])

print(s.str.strip().tolist())
print(s.str.lower().tolist())
print(s.str.upper().tolist())
print(s.str.replace('_', ' ', regex=False).tolist())
print(s.str.contains('thanks', case=False, na=False).tolist())
print(s.str.len().tolist())
print(s.str.split('_').tolist())
print(s.str.startswith('T', na=False).tolist())

['Hello_World', 'THANKS-you', 'yes!!', None]
[' hello_world ', 'thanks-you', 'yes!!', None]
[' HELLO_WORLD ', 'THANKS-YOU', 'YES!!', None]
[' Hello World ', 'THANKS-you', 'yes!!', None]
[False, True, False, False]
[13.0, 10.0, 5.0, nan]
[[' Hello', 'World '], ['THANKS-you'], ['yes!!'], None]
[False, True, False, False]


In [109]:
# Regex-powered extraction — pulling structured data out of unstructured strings
filenames = pd.Series(['gesture_hello_015.mp4', 'gesture_thanks_018.mp4'])
extracted = filenames.str.extract(r'gesture_(?P<label>\w+)_(?P<frame_count>\d+)\.mp4')
extracted

,label,frame_count
0,hello,015
1,thanks,018


In [110]:
# Chaining string methods (a very common real pipeline for cleaning raw labels)
raw_labels = pd.Series([' Hello!! ', 'THANKS.', ' yes '])
clean_labels = (
    raw_labels
    .str.strip()
    .str.lower()
    .str.replace(r'[^\w\s]', '', regex=True)
)
clean_labels

,0
0,hello
1,thanks
2,yes


## 17. Date and Time Handling (`.dt` accessor, resampling)

Timestamps are common in any logged data (session start times, frame
capture times). Pandas has first-class datetime support built on NumPy's
`datetime64`, plus a full time-series toolkit (`resample`, `rolling`,
`shift`) that is heavily used for sequential/streaming data.


In [111]:
dates = pd.to_datetime(['2025-01-01 10:00:00', '2025-01-01 10:05:30', '2025-01-02 09:00:00'])
s = pd.Series(dates)

print(s.dt.year.tolist())
print(s.dt.month.tolist())
print(s.dt.day.tolist())
print(s.dt.hour.tolist())
print(s.dt.day_name().tolist())
print(s.dt.dayofweek.tolist())

[2025, 2025, 2025]
[1, 1, 1]
[1, 1, 2]
[10, 10, 9]
['Wednesday', 'Wednesday', 'Thursday']
[2, 2, 3]


In [112]:
# date_range — generate a sequence of timestamps (useful for building synthetic time indexes)
pd.date_range(start='2025-01-01', periods=5, freq='D')

DatetimeIndex(['2025-01-01', '2025-01-02', '2025-01-03', '2025-01-04',
               '2025-01-05'],
              dtype='datetime64[ns]', freq='D')

In [113]:
# Timedeltas — durations, and arithmetic between timestamps
t0 = pd.Timestamp('2025-01-01 10:00:00')
t1 = pd.Timestamp('2025-01-01 10:05:30')
duration = t1 - t0
print(duration, type(duration))
print(t0 + pd.Timedelta(minutes=30))

0 days 00:05:30 <class 'pandas._libs.tslibs.timedeltas.Timedelta'>
2025-01-01 10:30:00


In [114]:
# Setting a DatetimeIndex and resampling — the pandas equivalent of SQL's time-bucketing
ts = pd.Series(
    [0.9, 0.85, 0.95, 0.88, 0.92],
    index=pd.date_range('2025-01-01', periods=5, freq='min')
)
print(ts)
print("\nResampled to 2-minute mean:")
print(ts.resample('2min').mean())

2025-01-01 00:00:00    0.90
2025-01-01 00:01:00    0.85
2025-01-01 00:02:00    0.95
2025-01-01 00:03:00    0.88
2025-01-01 00:04:00    0.92
Freq: min, dtype: float64

Resampled to 2-minute mean:
2025-01-01 00:00:00    0.875
2025-01-01 00:02:00    0.915
2025-01-01 00:04:00    0.920
Freq: 2min, dtype: float64


In [115]:
# shift() — lag/lead values, foundational for building time-series features (e.g. "confidence 1 step ago")
ts.shift(1)

,0
2025-01-01 00:00:00,NaN
2025-01-01 00:01:00,0.90
2025-01-01 00:02:00,0.85
2025-01-01 00:03:00,0.95
2025-01-01 00:04:00,0.88


## 18. Categorical Data

The `category` dtype stores repeated string values as integer codes behind
the scenes. For a column like `gesture` with a fixed, small set of possible
labels, this dramatically reduces memory use and speeds up groupby/sorting
compared to plain `object`/`string` dtype.


In [116]:
gestures = pd.Series(['Hello', 'Thanks', 'Hello', 'Yes', 'Hello'] * 1000)

print("As object dtype:", gestures.memory_usage(deep=True), "bytes")
gestures_cat = gestures.astype('category')
print("As category dtype:", gestures_cat.memory_usage(deep=True), "bytes")

As object dtype: 269132 bytes
As category dtype: 5401 bytes


In [117]:
# Ordered categories — needed when the categories have a natural order (e.g. confidence bands)
band = pd.Series(['low', 'high', 'medium', 'low', 'high'])
band_cat = pd.Categorical(band, categories=['low', 'medium', 'high'], ordered=True)
band_cat.sort_values() if hasattr(band_cat, 'sort_values') else sorted(band_cat)

['low', 'low', 'medium', 'high', 'high']
Categories (3, object): ['low' < 'medium' < 'high']

In [118]:
df = pd.DataFrame({'band': band_cat})
df.sort_values('band')   # sorts in the DEFINED category order, not alphabetically

,band
0,low
3,low
2,medium
1,high
4,high


## 19. Window Functions (`rolling`, `expanding`, `ewm`)

Window functions compute a statistic over a sliding subset of the data —
essential for smoothing noisy sequential signals, such as raw per-frame
hand-landmark confidence scores in a gesture-recognition pipeline.


In [119]:
confidence_stream = pd.Series([0.7, 0.9, 0.6, 0.95, 0.5, 0.99, 0.4, 0.98])

# rolling — fixed-size sliding window
print("3-frame rolling mean:\n", confidence_stream.rolling(window=3).mean())

3-frame rolling mean:
 0         NaN
1         NaN
2    0.733333
3    0.816667
4    0.683333
5    0.813333
6    0.630000
7    0.790000
dtype: float64


In [120]:
# expanding — window grows to include everything seen so far (cumulative statistic)
print("Expanding mean:\n", confidence_stream.expanding().mean())

Expanding mean:
 0    0.700000
1    0.800000
2    0.733333
3    0.787500
4    0.730000
5    0.773333
6    0.720000
7    0.752500
dtype: float64


In [121]:
# ewm — exponentially weighted mean, gives more weight to recent values.
# Often preferred over a simple rolling mean for real-time smoothing because
# it reacts faster to recent changes while still damping noise.
print("Exponentially weighted mean:\n", confidence_stream.ewm(span=3).mean())

Exponentially weighted mean:
 0    0.700000
1    0.833333
2    0.700000
3    0.833333
4    0.661290
5    0.828254
6    0.612441
7    0.796941
dtype: float64


**Practical note for this kind of gesture-recognition pipeline:** a rolling
or exponentially weighted average over the last few frames' prediction
confidence is a standard, lightweight technique to reduce flicker between
classes when the model's raw per-frame output is noisy — before committing
to a final predicted label for display or speech synthesis.


## 20. Combining and Updating DataFrames


In [122]:
# combine_first — fill NaNs in one DataFrame using values from another, aligned by index
primary = pd.DataFrame({'confidence': [0.9, np.nan, 0.8]}, index=['g1', 'g2', 'g3'])
backup = pd.DataFrame({'confidence': [np.nan, 0.75, 0.85]}, index=['g1', 'g2', 'g3'])
primary.combine_first(backup)

,confidence
g1,0.90
g2,0.75
g3,0.80


In [123]:
# update() — overwrite values in place from another DataFrame, aligned by index/columns
main = pd.DataFrame({'confidence': [0.9, 0.8, 0.7]}, index=['g1', 'g2', 'g3'])
corrections = pd.DataFrame({'confidence': [0.95, np.nan, np.nan]}, index=['g1', 'g2', 'g3'])
main.update(corrections)   # only non-NaN values from 'corrections' overwrite 'main'
main

,confidence
g1,0.95
g2,0.80
g3,0.70


## 21. Performance Optimization

For ML pipelines processing large gesture-landmark datasets, performance
mistakes compound quickly. The most impactful fixes, roughly in order of
payoff:


In [124]:
import time

n = 200_000
df = pd.DataFrame({'a': np.random.rand(n), 'b': np.random.rand(n)})

# SLOW: row-wise Python-level apply
start = time.time()
_ = df.apply(lambda row: row['a'] + row['b'], axis=1)
t_apply = time.time() - start

# FAST: vectorized addition
start = time.time()
_ = df['a'] + df['b']
t_vectorized = time.time() - start

print(f"apply(axis=1):  {t_apply:.4f}s")
print(f"vectorized:     {t_vectorized:.4f}s")
print(f"speedup: {t_apply / t_vectorized:.0f}x")

apply(axis=1):  2.0913s
vectorized:     0.0011s
speedup: 1855x


In [125]:
# Downcasting numeric dtypes reduces memory footprint substantially — important when
# loading large landmark datasets (e.g. 21 hand landmarks x, y, z per frame) into RAM.
df_big = pd.DataFrame({'x': np.random.rand(100_000).astype('float64')})
print("float64 memory:", df_big.memory_usage(deep=True).sum(), "bytes")

df_big['x'] = pd.to_numeric(df_big['x'], downcast='float')
print("downcast float memory:", df_big.memory_usage(deep=True).sum(), "bytes")

float64 memory: 800132 bytes
downcast float memory: 400132 bytes


In [126]:
# eval() / query() use a more memory-efficient evaluation engine for large DataFrames,
# avoiding intermediate temporary arrays that plain Python expressions create.
df = pd.DataFrame({'a': np.random.rand(1_000_000), 'b': np.random.rand(1_000_000)})

start = time.time()
_ = df['a'] + df['b']
t_normal = time.time() - start

start = time.time()
_ = df.eval('a + b')
t_eval = time.time() - start

print(f"normal: {t_normal:.4f}s, eval: {t_eval:.4f}s")
# On very large DataFrames eval()/query() typically win; on small ones the
# overhead of the expression engine can make them slower — always benchmark
# on your actual data size rather than assuming.

normal: 0.0029s, eval: 0.0207s


**Performance checklist for large datasets:**

| Technique | When to use | Risk if skipped |
|---|---|---|
| Vectorize instead of `.apply()`/loops | Always, when a vectorized equivalent exists | 10–100x slower processing on large datasets |
| `dtype` downcasting (`float32`, `int8`/`int16`, `category`) | Numeric/categorical columns with known ranges | Excess RAM use, may not fit in memory at all |
| `usecols` / `dtype` at read time | Reading large CSVs | Wastes time/memory parsing unused columns |
| `chunksize` for reading | Files larger than available RAM | `MemoryError` crash |
| Parquet over CSV | Any dataset that will be read more than once | Slower repeated reads, dtype loss on each round-trip |
| `df.eval()` / `df.query()` | Very large DataFrames with complex expressions | Unnecessary intermediate array allocation |
| Avoid growing a DataFrame row-by-row in a loop (`pd.concat` in a loop) | Building datasets incrementally | O(n²) behavior — collect rows in a list, build the DataFrame once at the end |


## 22. Plotting with Pandas

Pandas provides a thin, convenient wrapper around Matplotlib accessible
directly via `.plot()`. For quick exploratory plots during model
development this is faster than writing raw Matplotlib code; for
publication-quality or highly customized figures, drop down to Matplotlib
or Seaborn directly.


In [127]:
import matplotlib
matplotlib.use('Agg')   # non-interactive backend, safe for headless execution
import matplotlib.pyplot as plt

df = pd.DataFrame({
    'gesture': ['Hello', 'Thanks', 'Yes', 'No', 'Please'],
    'sample_count': [420, 380, 310, 295, 260]
})

ax = df.plot(x='gesture', y='sample_count', kind='bar', legend=False, title='Samples per gesture class')
ax.set_ylabel('Sample count')
plt.tight_layout()
plt.savefig('bar_plot.png')
plt.close()
print("Saved bar_plot.png")

Saved bar_plot.png


In [128]:
ts = pd.Series(np.random.rand(50).cumsum(), index=pd.date_range('2025-01-01', periods=50, freq='D'))
ax = ts.plot(kind='line', title='Cumulative training progress metric')
plt.tight_layout()
plt.savefig('line_plot.png')
plt.close()
print("Saved line_plot.png")

Saved line_plot.png


In [129]:
# Other common kind= values: 'hist', 'box', 'scatter', 'area', 'pie', 'kde'
df_numeric = pd.DataFrame({'confidence': np.random.beta(8, 2, 500)})
ax = df_numeric.plot(kind='hist', bins=20, title='Distribution of prediction confidence')
plt.tight_layout()
plt.savefig('hist_plot.png')
plt.close()
print("Saved hist_plot.png")

Saved hist_plot.png


## 23. Styling DataFrames

`.style` produces rich HTML-rendered tables inside Jupyter — useful for
visually auditing a results table (e.g. highlighting low-confidence
predictions) without leaving the notebook.


In [130]:
df = pd.DataFrame({
    'gesture': ['Hello', 'Thanks', 'Yes', 'No'],
    'confidence': [0.98, 0.62, 0.87, 0.45]
})

(
    df.style
    .background_gradient(subset=['confidence'], cmap='RdYlGn')
    .format({'confidence': '{:.1%}'})
    .set_caption('Prediction confidence by gesture')
)

,gesture,confidence
0,Hello,98.0%
1,Thanks,62.0%
2,Yes,87.0%
3,No,45.0%


In [131]:
# Highlighting values below a threshold — flags predictions that may need review
def highlight_low_confidence(val):
    return 'background-color: #ffcccc' if isinstance(val, float) and val < 0.7 else ''

df.style.map(highlight_low_confidence, subset=['confidence'])

,gesture,confidence
0,Hello,0.980000
1,Thanks,0.620000
2,Yes,0.870000
3,No,0.450000


## 24. Options, Settings, and Display Configuration


In [132]:
# Display options — control how much pandas prints without truncation
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 120)
pd.set_option('display.float_format', '{:.3f}'.format)

print(pd.get_option('display.max_rows'))

# Reset a single option back to default
pd.reset_option('display.max_rows')

100


In [133]:
# option_context — temporarily change display settings for a single block only,
# without affecting global state
with pd.option_context('display.max_rows', 5, 'display.float_format', '{:.2f}'.format):
    print(pd.DataFrame({'x': np.random.rand(10)}))

      x
0  0.81
1  0.69
..  ...
8  0.64
9  0.97

[10 rows x 1 columns]


## 25. Quick Reference Cheat-Sheet

| Task | Method |
|---|---|
| Load CSV / Excel / JSON / Parquet | `pd.read_csv/read_excel/read_json/read_parquet` |
| Save to file | `df.to_csv/to_excel/to_json/to_parquet` |
| First look at data | `df.head()`, `df.info()`, `df.describe()` |
| Select column(s) | `df['col']`, `df[['c1','c2']]` |
| Select by label | `df.loc[rows, cols]` |
| Select by position | `df.iloc[rows, cols]` |
| Filter rows | `df[df['col'] > x]`, `df.query('col > x')` |
| Add/modify column | `df['new'] = ...` |
| Drop rows/columns | `df.drop(index=..., columns=...)` |
| Missing values | `df.isna()`, `df.dropna()`, `df.fillna()` |
| Remove duplicates | `df.drop_duplicates()` |
| Sort | `df.sort_values()`, `df.sort_index()` |
| Group and aggregate | `df.groupby('key').agg(...)` |
| Reshape wide->long | `df.melt()` |
| Reshape long->wide | `df.pivot()`, `pd.pivot_table()` |
| Combine DataFrames | `pd.merge()`, `pd.concat()`, `df.join()` |
| String cleaning | `df['col'].str.strip().str.lower()` |
| Date parts | `df['col'].dt.year`, `.dt.month`, etc. |
| Rolling/smoothed stats | `df['col'].rolling(n).mean()` |
| Memory-efficient dtype | `df['col'].astype('category')`, `pd.to_numeric(..., downcast=...)` |
| Quick plot | `df.plot(kind='bar'/'line'/'hist'/'scatter')` |

---

